In [1]:
# Install necessary libraries
!pip install -q xgboost ipywidgets pandas numpy scikit-learn matplotlib

import numpy as np
import pandas as pd
import datetime
import xgboost as xgb
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Simulate historical traffic and incident data with dates (2025 - 2026)
np.random.seed(42)
n_samples = 3500

locations = [
    {"name": "Longcommon Rd & E Burlington St", "lat": 41.8295, "lon": -87.8182},
    {"name": "Riverside Rd & Bloomingbank Rd", "lat": 41.8312, "lon": -87.8225},
    {"name": "First Ave & 31st St Intersection", "lat": 41.8385, "lon": -87.8341},
    {"name": "Woodside Rd & Harlem Ave", "lat": 41.8250, "lon": -87.8050},
    {"name": "Pasadena Dr & Ridgewood Rd", "lat": 41.8270, "lon": -87.8150},
    {"name": "Delaplaine Rd & Quincy St", "lat": 41.8340, "lon": -87.8190}
]

date_start = datetime.date(2025, 1, 1)
date_end = datetime.date(2026, 12, 31)
delta_days = (date_end - date_start).days

data_rows = []
for _ in range(n_samples):
    loc = np.random.choice(locations)
    random_days = np.random.randint(0, delta_days)
    record_date = date_start + datetime.timedelta(days=random_days)

    hour = np.random.randint(0, 24)
    day_of_week = record_date.weekday()
    temperature = np.random.normal(15, 10)
    precipitation = np.random.choice([0.0, 0.5, 2.5, 10.0], p=[0.7, 0.2, 0.08, 0.02])
    traffic_flow = np.random.randint(100, 1500)
    is_roadwork = np.random.choice([0, 1], p=[0.85, 0.15])

    risk_score = (
        (1.5 if (7 <= hour <= 9 or 17 <= hour <= 19) else 0.5) * 0.3 +
        (1.2 if day_of_week < 5 else 0.8) * 0.1 +
        (1.4 if precipitation > 2.0 else 1.0) * 0.2 +
        (traffic_flow / 1000.0) * 0.3 +
        (1.5 if is_roadwork == 1 else 1.0) * 0.1
    )
    risk_score += np.random.normal(0, 0.1)
    target = 1 if risk_score > 1.15 else 0

    data_rows.append({
        'date': record_date,
        'location_name': loc['name'],
        'hour': hour,
        'day_of_week': day_of_week,
        'temperature': round(temperature, 1),
        'precipitation': precipitation,
        'traffic_flow': traffic_flow,
        'is_roadwork': is_roadwork,
        'target_incident': target
    })

df = pd.DataFrame(data_rows)

# 2. Interactive Widgets Setup (Including Period Selector)
period_widget = widgets.Dropdown(
    options=[
        ('All Periods (2025-2026)', 'ALL'),
        ('Year 2025 Only', '2025'),
        ('Year 2026 Only', '2026'),
        ('Winter Season (Dec-Feb)', 'WINTER'),
        ('Summer Season (Jun-Aug)', 'SUMMER')
    ],
    value='ALL',
    description='Historical Period:',
    style={'description_width': '140px'}
)

location_widget = widgets.Dropdown(options=[(loc['name'], i) for i, loc in enumerate(locations)], value=0, description='Intersection:', style={'description_width': '140px'})
hour_widget = widgets.IntSlider(value=8, min=0, max=23, step=1, description='Hour of Day:', style={'description_width': '140px'})
day_widget = widgets.Dropdown(options=[('Monday', 0), ('Tuesday', 1), ('Wednesday', 2), ('Thursday', 3), ('Friday', 4), ('Saturday', 5), ('Sunday', 6)], value=0, description='Day of Week:', style={'description_width': '140px'})
temp_widget = widgets.FloatSlider(value=18.0, min=-15.0, max=35.0, step=0.5, description='Temp (°C):', style={'description_width': '140px'})
precip_widget = widgets.Dropdown(options=[('None', 0.0), ('Light Rain', 0.5), ('Heavy Rain', 2.5), ('Storm / Snow', 10.0)], value=0.0, description='Precipitation:', style={'description_width': '140px'})
flow_widget = widgets.IntSlider(value=800, min=100, max=1500, step=50, description='Traffic (veh/h):', style={'description_width': '140px'})
roadwork_widget = widgets.Checkbox(value=False, description='Roadwork Active')

out = widgets.Output()

def update_simulation_no_map(b=None):
    with out:
        clear_output(wait=True)

        # Filter dataframe based on selected historical period
        p_val = period_widget.value
        if p_val == '2025':
            filtered_df = df[df['date'].apply(lambda d: d.year == 2025)]
        elif p_val == '2026':
            filtered_df = df[df['date'].apply(lambda d: d.year == 2026)]
        elif p_val == 'WINTER':
            filtered_df = df[df['date'].apply(lambda d: d.month in [12, 1, 2])]
        elif p_val == 'SUMMER':
            filtered_df = df[df['date'].apply(lambda d: d.month in [6, 7, 8])]
        else:
            filtered_df = df.copy()

        if len(filtered_df) < 50:
            filtered_df = df.copy() # fallback if subset is too small

        # Train XGBoost on filtered period data
        features = ['hour', 'day_of_week', 'temperature', 'precipitation', 'traffic_flow', 'is_roadwork']
        X = filtered_df[features]
        y = filtered_df['target_incident']

        model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
        model.fit(X, y)

        selected_loc = locations[location_widget.value]
        input_data = pd.DataFrame([{
            'hour': hour_widget.value,
            'day_of_week': day_widget.value,
            'temperature': temp_widget.value,
            'precipitation': precip_widget.value,
            'traffic_flow': flow_widget.value,
            'is_roadwork': int(roadwork_widget.value)
        }])

        probability = model.predict_proba(input_data)[0][1] * 100

        print(f"📊 Selected Period: {period_widget.label} ({len(filtered_df)} records analyzed)")
        print(f"📍 Location: {selected_loc['name']} (Riverside, Chicago)")
        print(f"🚨 Predicted Hotspot Probability: {probability:.1f}%")
        if probability > 60:
            print("⚠️ WARNING: High risk! Patrol unit dispatch advised.")
        elif probability > 30:
            print("⚡ Moderate risk level.")
        else:
            print("✅ Stable traffic conditions.")

        loc_history = filtered_df[filtered_df['location_name'] == selected_loc['name']]
        hist_pct = (loc_history['target_incident'].mean() * 100) if len(loc_history) > 0 else 0.0
        print(f"📈 Historical incident frequency for this period: {hist_pct:.1f}%\n")

        # Plot feature importance
        fig, ax = plt.subplots(figsize=(8, 3))
        xgb.plot_importance(model, ax=ax, max_num_features=5, importance_type='weight', color='darkorange')
        ax.set_title(f"XGBoost Feature Importance ({period_widget.label})")
        plt.tight_layout()
        plt.show()

run_button = widgets.Button(description='Run Analysis', button_style='success', icon='play')
run_button.on_click(update_simulation_no_map)

display(widgets.VBox([
    widgets.HTML("<h3>Traffic Hotspot Predictor - Text Mode & Period Selector</h3>"),
    period_widget,
    location_widget,
    widgets.HBox([hour_widget, day_widget]),
    widgets.HBox([temp_widget, precip_widget]),
    widgets.HBox([flow_widget, roadwork_widget]),
    run_button,
    widgets.HTML("<hr>")
], layout=widgets.Layout(width='100%')), out)

update_simulation_no_map()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 18.7 MB/s eta 0:00:00


Output()